In [1]:
import websocket
import json
import time
import pyarrow as pa
import pyarrow.parquet as pq
from datetime import datetime
import threading
import os

url = "wss://ws.api.prod.paradex.trade/v1"
market = "ETH-USD-PERP"
channel = f"bbo.{market}"
file_path = "paradex_orderbook.parquet"
subscription_id = 1  # Arbitrary ID for the subscription request

def on_message(ws, message):
    try:
        data = json.loads(message)
        # print(f"Received: {message}")
        
        # Handle subscription updates (ignores confirmation)
        if "method" in data and data["method"] == "subscription" and "params" in data:
            params = data["params"]
            if "channel" in params and params["channel"] == channel and "data" in params:
                inner_data = params["data"]
                bid_px = inner_data.get("bid")
                bid_sz = inner_data.get("bid_size")
                ask_px = inner_data.get("ask")
                ask_sz = inner_data.get("ask_size")
                
                if bid_px and bid_sz and ask_px and ask_sz:
                    timestamp = datetime.now().strftime("%Y-%m-%d %H:%M:%S.%f")[:-3]  # Milliseconds
                    para_best_bid_px = float(bid_px)
                    para_best_bid_sz = float(bid_sz)
                    para_best_ask_px = float(ask_px)
                    para_best_ask_sz = float(ask_sz)

                    # Create a single-row pyarrow Table
                    new_data = {
                        "timestamp": [timestamp],
                        "para_best_bid_px": [para_best_bid_px],
                        "para_best_bid_sz": [para_best_bid_sz],
                        "para_best_ask_px": [para_best_ask_px],
                        "para_best_ask_sz": [para_best_ask_sz]
                    }
                    new_table = pa.Table.from_pydict(new_data)

                    # Append to existing Parquet file (or create if not exists)
                    if os.path.exists(file_path):
                        existing_table = pq.read_table(file_path)
                        updated_table = pa.concat_tables([existing_table, new_table])
                    else:
                        updated_table = new_table
                    
                    pq.write_table(updated_table, file_path)
                    print(f"Appended at {timestamp}: Bid {para_best_bid_px}/{para_best_bid_sz}, Ask {para_best_ask_px}/{para_best_ask_sz}")
                    time.sleep(0.09)
    except Exception as e:
        print(f"Message error: {e}")

def on_error(ws, error):
    print(f"Error: {error}")

def on_close(ws, close_status_code, close_msg):
    ws.run_forever()
    print(f"Closed: {close_msg}")

def on_open(ws):
    print("Connected. Subscribing...")
    subscribe_msg = {
        "jsonrpc": "2.0",
        "method": "subscribe",
        "params": {
            "channel": channel
        },
        "id": subscription_id
    }
    ws.send(json.dumps(subscribe_msg))


ws = websocket.WebSocketApp(
    url,
    on_open=on_open,
    on_message=on_message,
    on_error=on_error,
    on_close=on_close,  
)

def run_forever_with_ping():
    ws.run_forever()

thread = threading.Thread(target=run_forever_with_ping)
thread.start()


# Keep main thread alive
try:
    while True:
        time.sleep(1)
except KeyboardInterrupt:
    ws.close()
    print("Stopping...")

Connected. Subscribing...
Appended at 2025-12-03 17:08:37.377: Bid 3048.71/6.5599, Ask 3048.95/0.7413
Appended at 2025-12-03 17:08:38.049: Bid 3048.71/6.5599, Ask 3048.96/5.4014
Appended at 2025-12-03 17:08:38.210: Bid 3048.71/6.5599, Ask 3048.97/0.321
Appended at 2025-12-03 17:08:38.308: Bid 3048.71/6.5599, Ask 3048.94/5.4014
Appended at 2025-12-03 17:08:38.407: Bid 3048.71/6.5599, Ask 3048.94/5.393
Appended at 2025-12-03 17:08:38.506: Bid 3048.71/6.5599, Ask 3048.94/5.3439
Appended at 2025-12-03 17:08:38.601: Bid 3048.66/0.02, Ask 3048.94/5.3439
Appended at 2025-12-03 17:08:38.700: Bid 3048.36/0.6653, Ask 3048.94/5.3439
Appended at 2025-12-03 17:08:38.800: Bid 3048.36/0.6653, Ask 3048.93/0.7413
Appended at 2025-12-03 17:08:38.903: Bid 3048.36/0.6653, Ask 3048.93/1.6995
Appended at 2025-12-03 17:08:39.006: Bid 3048.36/0.6653, Ask 3048.93/0.9582
Appended at 2025-12-03 17:08:39.110: Bid 3048.43/0.328, Ask 3048.93/0.9582
Appended at 2025-12-03 17:08:39.211: Bid 3048.67/6.5601, Ask 3048.9

Connected. Subscribing...
Appended at 2025-12-03 18:37:27.581: Bid 3060.13/5.4014, Ask 3060.37/0.3467
Appended at 2025-12-03 18:37:27.690: Bid 3060.14/6.5355, Ask 3060.37/0.3467
Appended at 2025-12-03 18:37:27.795: Bid 3060.15/5.4014, Ask 3060.37/0.3467
Appended at 2025-12-03 18:37:27.935: Bid 3060.15/5.393, Ask 3060.37/0.3467
Appended at 2025-12-03 18:37:28.044: Bid 3060.15/5.3847, Ask 3060.37/0.3467
Appended at 2025-12-03 18:37:28.158: Bid 3060.15/5.3763, Ask 3060.37/0.3467
Appended at 2025-12-03 18:37:28.271: Bid 3060.16/6.5353, Ask 3060.37/0.3467
Appended at 2025-12-03 18:37:28.412: Bid 3060.16/6.527, Ask 3060.37/0.3467
Appended at 2025-12-03 18:37:28.527: Bid 3060.16/6.5187, Ask 3060.37/0.3467
Appended at 2025-12-03 18:37:28.636: Bid 3060.17/5.4014, Ask 3060.37/0.3467
Appended at 2025-12-03 18:37:28.742: Bid 3060.18/6.5352, Ask 3060.37/0.3467
Appended at 2025-12-03 18:37:28.858: Bid 3060.18/6.5352, Ask 3060.37/0.02
Appended at 2025-12-03 18:37:28.975: Bid 3060.18/9.776, Ask 3060.3